# Temporal graph inspection

Build and inspect the hand-checkable CPU fixture rather than loading the full ICEWS14 source. The fixture has two directed two-hop paths, distinct event dates, and a missing seed entity. The cutoff traversal below is a notebook-only inspection; production temporal filtering is a later task.

In [1]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print("GPU:", props.name)
    print(f"VRAM: {props.total_memory / 2**30:.1f} GiB")

PyTorch: 2.14.1+cu130
CUDA available: True
GPU: Tesla T4
VRAM: 14.6 GiB


In [3]:
%pip install -q transformers accelerate bitsandbytes

In [4]:
import time
import torch
import transformers
import bitsandbytes
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

assert torch.cuda.is_available(), "CUDA is not available"
print("GPU:", torch.cuda.get_device_name(0))
print("PyTorch:", torch.__version__, "| Transformers:", transformers.__version__)
print("bitsandbytes:", bitsandbytes.__version__)

model_id = "Qwen/Qwen2.5-7B-Instruct"
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=quant_config,
    device_map="auto",
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
)

messages = [{"role": "user", "content": "Reply with a brief greeting."}]
prompt = tokenizer.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

torch.cuda.reset_peak_memory_stats()
start = time.perf_counter()
with torch.inference_mode():
    output = model.generate(**inputs, max_new_tokens=32, do_sample=False)
torch.cuda.synchronize()

print("Response:", tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True))
print(f"Inference seconds: {time.perf_counter() - start:.2f}")
print(f"Peak allocated VRAM: {torch.cuda.max_memory_allocated() / 2**30:.2f} GiB")

GPU: Tesla T4
PyTorch: 2.14.1+cu130 | Transformers: 5.22.1
bitsandbytes: 0.50.2


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

... (4 outputs left out)

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Response: Hello! How can I assist you today?
Inference seconds: 1.67
Peak allocated VRAM: 5.19 GiB


In [ ]:
import json
from datetime import date
from pathlib import Path

from src.graph.graph_builder import build_temporal_graph

In [ ]:
fixture_relative_path = Path("tests") / "fixtures" / "temporal_fixture.json"
fixture_path = next(
    (
        base / fixture_relative_path
        for base in (Path.cwd(), *Path.cwd().parents)
        if (base / fixture_relative_path).is_file()
    ),
    None,
)
if fixture_path is None:
    raise FileNotFoundError(f"Could not locate {fixture_relative_path} from {Path.cwd()}")

fixture = json.loads(fixture_path.read_text(encoding="utf-8"))
graph = build_temporal_graph(fixture["facts"])
print(f"Fixture: {fixture['fixture_id']}")
print(f"Entities: {len(graph.entity_ids)} | relations: {len(graph.relation_ids)} | facts: {graph.fact_count}")

In [ ]:
print("Dated directed facts")
for fact in graph.facts:
    print(
        f"{fact['timestamp']}  {fact['subject_id']} "
        f"-[{fact['relation_id']}]-> {fact['object_id']} "
        f"(fact_id={fact['fact_id']})"
    )

In [ ]:
seed = "entity_a"
print(f"Outgoing from {seed}:")
for fact in graph.outgoing_facts(seed):
    print(f"  {fact['fact_id']}: {fact['relation_id']} -> {fact['object_id']} ({fact['timestamp']})")

print("\nIncoming to entity_c:")
for fact in graph.incoming_facts("entity_c"):
    print(f"  {fact['fact_id']}: {fact['subject_id']} -> entity_c ({fact['timestamp']})")

Trace each fixture query using the graph's outgoing-fact index. Dates are compared inclusively here to make the fixture's boundary behavior visible; this small traversal is not a replacement for the planned retrieval module.

In [ ]:
def trace_paths(seed_entity_id, relation_sequence, cutoff_date):
    cutoff = date.fromisoformat(cutoff_date)
    paths = [(seed_entity_id, ())]
    for relation_id in relation_sequence:
        next_paths = []
        for entity_id, fact_ids in paths:
            for fact in graph.outgoing_facts(entity_id, relation_id):
                if date.fromisoformat(fact["timestamp"]) <= cutoff:
                    next_paths.append((fact["object_id"], fact_ids + (fact["fact_id"],)))
        paths = next_paths
    return paths


for query in fixture["queries"]:
    paths = trace_paths(
        query["seed_entity_id"], query["relation_sequence"], query["cutoff_date"]
    )
    path_ids = sorted([list(fact_ids) for _, fact_ids in paths])
    answer_ids = sorted({entity_id for entity_id, _ in paths})
    assert path_ids == sorted(query["expected_paths"]), query["query_id"]
    assert answer_ids == sorted(query["expected_answer_ids"]), query["query_id"]
    print(f"{query['query_id']} (cutoff {query['cutoff_date']}):")
    print(f"  paths={path_ids} answers={answer_ids}")